In [2]:
# @title 1. Cài đặt & Mount Drive
!pip install -q transformers peft datasets evaluate scikit-learn accelerate psutil

import os
import torch
import psutil
import pandas as pd
import numpy as np
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')
SAVE_PATH = '/content/drive/My Drive/SLM_Research/IMDB_Falcon1B_PromptTuning'
if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print("✅ Phần 1: Cài đặt hoàn tất!")

Mounted at /content/drive
Using device: cuda
✅ Phần 1: Cài đặt hoàn tất!


In [3]:
# @title 2. Load Data (IMDB) & Tokenizer
from datasets import load_dataset
from transformers import AutoTokenizer

print("--- Loading IMDB Dataset ---")
dataset = load_dataset("imdb")
MODEL_NAME = 'tiiuae/falcon-rw-1b'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize_function(examples):
    # Cắt gọt xuống 512 tokens
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

print("--- Tokenizing (Mất khoảng 1-2 phút) ---")
tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets.set_format("torch")

# Dataset gốc: Train (25k), Test (25k)
dataset_train = tokenized_datasets["train"]
dataset_val = tokenized_datasets["test"]

print(f"Train size: {len(dataset_train)} | Test size: {len(dataset_val)}")
print("✅ Phần 2: Data sẵn sàng!")

--- Loading IMDB Dataset ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

...

--- Tokenizing (Mất khoảng 1-2 phút) ---


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Train size: 25000 | Test size: 25000
✅ Phần 2: Data sẵn sàng!


In [4]:
# @title 3. Model Setup (Falcon + Prompt Tuning)
from transformers import AutoModelForSequenceClassification
from peft import get_peft_model, PromptTuningConfig, TaskType, PromptTuningInit

print("--- Loading Falcon-1B (FP16) ---")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    torch_dtype=torch.float16
)
model.config.pad_token_id = tokenizer.pad_token_id
model.gradient_checkpointing_enable()

# --- CẤU HÌNH PROMPT TUNING CHO FALCON ---
# Phải khai báo tường minh cấu trúc model để tránh lỗi PEFT không nhận diện được Falcon
peft_config = PromptTuningConfig(
    task_type=TaskType.SEQ_CLS,
    num_virtual_tokens=10,                      # Số lượng token học thêm
    prompt_tuning_init=PromptTuningInit.RANDOM, # Khởi tạo ngẫu nhiên
    num_layers=model.config.num_hidden_layers,
    token_dim=model.config.hidden_size,
    num_attention_heads=model.config.num_attention_heads,
    num_transformer_submodules=1
)

model = get_peft_model(model, peft_config)

# --- FIX LỖI "Unscale FP16 Gradients" ---
# Prompt Embeddings cần phải là Float32
print("--- Casting Prompt Embeddings to Float32 ---")
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

model.to(device)
model.print_trainable_parameters()
print("✅ Phần 3: Model Prompt Tuning sẵn sàng!")

--- Loading Falcon-1B (FP16) ---


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/2.62G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.62G [00:00<?, ?B/s]

Some weights of FalconForSequenceClassification were not initialized from the model checkpoint at tiiuae/falcon-rw-1b and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


--- Casting Prompt Embeddings to Float32 ---
trainable params: 24,576 || all params: 1,311,653,888 || trainable%: 0.0019
✅ Phần 3: Model Prompt Tuning sẵn sàng!


In [5]:
# @title 4. Smart Training (Prompt Tuning - Save by Epochs)
import time
import numpy as np
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from transformers.trainer_utils import get_last_checkpoint
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from scipy.special import softmax

# 1. Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple): logits = logits[0]
    predictions = np.argmax(logits, axis=-1)
    probs = softmax(logits, axis=1)[:, 1]

    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    roc_auc = roc_auc_score(labels, probs)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall, 'roc_auc': roc_auc}

# 2. Config (Updated for Epoch Strategy)
training_args = TrainingArguments(
    output_dir=SAVE_PATH,
    num_train_epochs=3,               # 3 Epochs
    per_device_train_batch_size=4,    # Batch 4
    gradient_accumulation_steps=1,
    gradient_checkpointing=False,

    # --- THAY ĐỔI: LƯU VÀ ĐÁNH GIÁ THEO EPOCH ---
    save_strategy="epoch",            # Lưu mỗi khi hết 1 epoch
    save_total_limit=1,               # Chỉ giữ 1 bản tốt nhất

    eval_strategy="epoch",            # Đánh giá mỗi khi hết 1 epoch
    # --------------------------------------------

    # QUAN TRỌNG: Prompt Tuning cần LR cao (3e-2)
    learning_rate=3e-2,
    warmup_steps=200,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
    report_to="none"
)

# 3. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_val,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# 4. Auto Resume
model.config.use_cache = False
print(f"--- Checking checkpoint: {SAVE_PATH} ---")
last_checkpoint = get_last_checkpoint(SAVE_PATH)

start_train_time = time.time()
if last_checkpoint:
    print(f"🔄 Resume from: {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("✨ Start New Training...")
    trainer.train()

training_time = time.time() - start_train_time
trainer.save_model(SAVE_PATH)
print("✅ Phần 4: Huấn luyện xong (Prompt Tuning - Saved by Epochs)!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 50256, 'bos_token_id': 50256}.


--- Checking checkpoint: /content/drive/My Drive/SLM_Research/IMDB_Falcon1B_PromptTuning ---
🔄 Resume from: /content/drive/My Drive/SLM_Research/IMDB_Falcon1B_PromptTuning/checkpoint-12500


FalconForSequenceClassification will not detect padding tokens in `inputs_embeds`. Results may be unexpected if using padding tokens in conjunction with `inputs_embeds.`


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Roc Auc
3,0.373400,0.367412,0.882480,0.882705,0.881017,0.884400,0.952470


✅ Phần 4: Huấn luyện xong (Prompt Tuning - Saved by Epochs)!


In [6]:
# @title 5. Final Report (IMDB - Full 10 Metrics)
import os
import time
import pandas as pd
import psutil
import torch

print("--- Đang đánh giá lần cuối trên tập Test IMDB ---")
# Quá trình này sẽ mất thời gian hơn SST-2
start_pred = time.time()
predictions_output = trainer.predict(dataset_val)
end_pred = time.time()

metrics = predictions_output.metrics
latency = ((end_pred - start_pred) / len(dataset_val)) * 1000  # ms/sample

# 2. Tính kích thước Prompt (Rất nhỏ)
adapter_bin = os.path.join(SAVE_PATH, 'adapter_model.bin')
adapter_safe = os.path.join(SAVE_PATH, 'adapter_model.safetensors')
size_mb = 0
if os.path.exists(adapter_bin): size_mb = os.path.getsize(adapter_bin)
elif os.path.exists(adapter_safe): size_mb = os.path.getsize(adapter_safe)
size_mb /= (1024**2)

# 3. Resources
process = psutil.Process(os.getpid())
ram_mb = process.memory_info().rss / (1024**2)
vram_mb = torch.cuda.max_memory_allocated() / (1024**2) if torch.cuda.is_available() else 0
curr_train_time = training_time if 'training_time' in locals() else 0.0

# 4. In Báo Cáo
print("\n====== REPORT: Falcon-1B + Prompt Tuning (IMDB) ======")
print(f"1. Classification Metrics:")
print(f"   - Accuracy:  {metrics.get('test_accuracy', 0):.4f}")
print(f"   - Precision: {metrics.get('test_precision', 0):.4f}")
print(f"   - Recall:    {metrics.get('test_recall', 0):.4f}")
print(f"   - F1-Score:  {metrics.get('test_f1', 0):.4f}")
print(f"   - ROC-AUC:   {metrics.get('test_roc_auc', 0):.4f}")

print(f"\n2. Efficiency Metrics:")
print(f"   - Training Time:      {curr_train_time:.2f} s")
print(f"   - Inference Latency:  {latency:.4f} ms/sample")
print(f"   - Adapter Size (Disk): {size_mb:.4f} MB")
print(f"   - Peak RAM Usage:     {ram_mb:.2f} MB")
print(f"   - Peak VRAM Usage:    {vram_mb:.2f} MB")

# 5. Lưu CSV
results_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC",
               "Training Time (s)", "Inference Latency (ms)", "Adapter Size (MB)",
               "Peak RAM (MB)", "Peak VRAM (MB)"],
    "Value": [
        metrics.get('test_accuracy', 0),
        metrics.get('test_precision', 0),
        metrics.get('test_recall', 0),
        metrics.get('test_f1', 0),
        metrics.get('test_roc_auc', 0),
        curr_train_time,
        latency,
        size_mb,
        ram_mb,
        vram_mb
    ]
})

results_file = os.path.join(SAVE_PATH, 'imdb_falcon_prompt_full_report.csv')
results_df.to_csv(results_file, index=False)
print(f"\n✅ Báo cáo đầy đủ đã được lưu tại: {results_file}")

--- Đang đánh giá lần cuối trên tập Test IMDB ---



====== REPORT: Falcon-1B + Prompt Tuning (IMDB) ======
1. Classification Metrics:
   - Accuracy:  0.8825
   - Precision: 0.8810
   - Recall:    0.8844
   - F1-Score:  0.8827
   - ROC-AUC:   0.9525

2. Efficiency Metrics:
   - Training Time:      8115.76 s
   - Inference Latency:  78.4961 ms/sample
   - Adapter Size (Disk): 0.0939 MB
   - Peak RAM Usage:     3693.54 MB
   - Peak VRAM Usage:    3061.09 MB

✅ Báo cáo đầy đủ đã được lưu tại: /content/drive/My Drive/SLM_Research/IMDB_Falcon1B_PromptTuning/imdb_falcon_prompt_full_report.csv
